 **Structured Solution for **Task 1** FOR Week 2 assignment.**

---

## Task 1 — The Three Tiers, and What a Warehouse Must Be

### 1. The Three Tiers

* **Tier 1 (Staging)**
* **One-line job:** Acts as a raw data landing zone and quality checkpoint.
* **PakFreight case example:** Importing `marine_shipments.csv`, `rail_shipments.xlsx`, and `road_shipments.csv` into separate staging tables without altering source files.
* **Must not happen here:** Aggregating data, joining across multiple source formats, or serving business reports.


* **Tier 2 (Storage / EDW / ODS)**
* **One-line job:** Maintains enterprise-wide integrated, historical data persistence.
* **PakFreight case example:** Storing consolidated shipment records that combine marine, rail, and road data into a unified schema.
* **Must not happen here:** Direct user reporting/querying or unfiltered raw ingestion.


* **Tier 3 (Presentation / Data Marts)**
* **One-line job:** Delivers business-ready, dimensional models (star schemas) optimized for BI tools.
* **PakFreight case example:** Creating a star schema for the Board or CFO to analyze mode profitability and freight costs.
* **Must not happen here:** Storing raw, untransformed source logs or applying direct operational updates.



---

### 2. Inmon's Four Properties Applied to PakFreight

* **Subject-Oriented**
* **Satisfied:** The warehouse is structured around core business entities like `Shipments` rather than isolated operational applications (marine, rail, road).
* **Broken:** Keeping separate tables strictly tethered to individual department spreadsheets instead of a unified subject domain.


* **Integrated**
* **Satisfied:** Standardizing divergent units (KG in marine, TONS in rail) into a single unified metric and normalizing currencies into `PKR`.
* **Broken:** Retaining conflicting schemas, mismatched status casing (`UPPERCASE`, `mixed-case`, `lowercase`), and mixed date formats (`dd/mm/yyyy` vs Excel serials) as separate unlinked islands.


* **Time-Variant**
* **Satisfied:** Retaining historical shipment records from 2023 onward with proper `ShipDate` values for multi-year trend analysis.
* **Broken:** Overwriting old shipment statuses or deleting records when a trip state updates.


* **Non-Volatile**
* **Satisfied:** Loading historical data as read-only once committed to the enterprise storage tier.
* **Broken:** Directly modifying or overwriting historical records in the warehouse when a road trip row contains a duplicate or correction, rather than appending a tracked update.



---

### 3. EDW or ODS Classification

* **a) A dispatcher needs truck positions every 2 minutes:** **ODS** (Operational Data Store) — requires near-real-time operational detail.
* **b) The CFO compares freight cost per tonne-km across three years:** **EDW** (Enterprise Data Warehouse) — requires integrated historical data for multi-year trend analysis.
* **c) A customer service agent checks whether consignment 45017 left today:** **ODS** — requires up-to-the-minute operational transaction status.
* **d) The board reviews mode profitability since 2023:** **EDW** — requires long-term, aggregated historical data across business units.

---

### 4. Independent vs. Dependent Data Marts

#### Architectural Diagrams

* **Independent (Siloed) Marts:**
```text
Marine File ──> Operations Mart (Silo 1)
Rail File   ──> Finance Mart    (Silo 2)
Road File   ──> Sales Mart      (Silo 3)

```


* **Dependent Data Marts (Kimball / Enterprise approach):**
```text
Sources ──> Tier 1 Staging ──> Tier 2 Enterprise Data Warehouse (EDW) ──┬──> Operations Mart
                                                                      ├──> Finance Mart
                                                                      └──> Sales Mart

```



#### Explanation

Independent data marts pull data directly from siloed source files without a central integrated warehouse. Because each department cleans, transforms, and defines metrics independently using its own rules, they arrive at conflicting business definitions and calculations. Consequently, querying the exact same enterprise question across these siloed marts produces three different answers.

---

### 5. Why Staging is Necessary

Raw files pass through staging to provide an isolated quality checkpoint, ensuring data lineage, auditability, and protection of source systems before any structural transformation or loading into the core warehouse model.

---

### 6. Overwriting Customer Addresses

Overwriting an address breaks Inmon's **non-volatile** and **time-variant** properties because historical states are destroyed rather than preserved. Instead, the warehouse should employ a Slowly Changing Dimension (SCD) pattern (such as SCD Type 2) to maintain historical accuracy by creating a new record version with effective date tracking.

 solution for **Task 2**.

---

## Task 2 — Inmon, Kimball, and the Lifecycle

### 1. Side-by-Side Comparison Table

| Dimension | Inmon (Top-Down) | Kimball (Bottom-Up) |
| --- | --- | --- |
| **Starting Point** | Enterprise-wide scope; builds a centralized normalized core first. | Individual business processes; builds dimensional data marts iteratively. |
| **Modelling Style** | Third Normal Form (3NF) for the Enterprise Data Warehouse (EDW). | Dimensional modeling (Star schema with fact and dimension tables). |
| **Time to First Value** | **Slow** (requires long upfront integration before data marts deliver value). | **Fast** (delivers value incrementally, one business process mart at a time). |
| **Risk** | High initial cost, complexity, and risk of analysis paralysis. | Risk of silo drift and inconsistent metrics if conformed dimensions aren't managed. |
| **Consistency Enforcement** | Enforced upfront at the EDW level through centralized data modeling. | Enforced horizontally across marts using **conformed dimensions** (Kimball Bus). |

---

### 2. Architectural Diagrams (CIF vs. Bus)

#### Inmon's Corporate Information Factory (CIF)

```text
[Source Systems] ──> [Staging] ──> [Normalized EDW (3NF)] ──> [Dependent Data Marts] ──> [BI / Reports]

```

* **Data Flow:** Raw source data lands in staging, is integrated and stored centrally in a normalized 3NF enterprise warehouse, and is then pushed down into dependent departmental marts.

#### Kimball's Dimensional Bus Architecture

```text
[Source Systems] ──> [Staging] ──> [Dimensional Data Marts (Star Schemas)] <──> [Conformed Dimensions] ──> [BI / Reports]

```

* **Data Flow:** Source data is transformed into star schemas per business process. Consistency across disparate marts is maintained by sharing reusable **conformed dimensions** (e.g., Date, Route).

---

### 3. Classification of Scenarios

1. **A bank must have one enterprise definition of "customer" before any reporting:** **Inmon** — Requires top-down enterprise normalization and master data integration first.
2. **A startup needs a sales dashboard live within six weeks:** **Kimball** — Bottom-up approach optimized for fast, incremental time-to-value.
3. **A hospital group with five systems and a three-year regulatory programme:** **Inmon** — Long-term, enterprise-wide structural integration program.
4. **PakFreight builds the Shipments mart first, with conformed Date and Route:** **Kimball** — Bottom-up dimensional approach focusing on a single business process first.
5. **A retailer with a normalised core warehouse feeding departmental marts:** **Inmon** — Classic top-down 3NF EDW feeding dependent marts.
6. **A telecom moving to a lakehouse with bronze, silver and gold layers:** **Hybrid** — Combines Inmon-style foundational integration/storage (silver) with Kimball-style business-level presentation marts (gold).
7. **A government census programme where the model must be agreed before loading:** **Inmon** — Rigorous top-down data governance and centralized schema design.
8. **An e-commerce team adding a Returns mart that reuses the existing Product dimension:** **Kimball** — Textbook dimensional bus expansion using conformed dimensions.
9. **A manufacturer who has built six marts and now cannot reconcile them:** **Kimball (unconformed) / Hybrid** — Symptom of bottom-up marts drifting apart without conformed dimensions; requires retrofitting a bus matrix.
10. **A bank integrating a newly acquired competitor's systems:** **Inmon** — Enterprise master data integration and normalization across disparate core backends.

---

### 4. Kimball Lifecycle for PakFreight Iteration 1 (Shipments)

* **Business Requirements:** Identifying core business metrics needed for shipments (origin, destination, weight in KG, cost in PKR, and delivery status across marine, rail, and road).
* **Dimensional Modelling:** Designing the Shipments star schema, comprising a central `Fact_Shipment` table surrounded by dimensions like `Dim_Date`, `Dim_Route`, and `Dim_SourceSystem`.
* **ETL Design and Development:** Building Power Query staging pipelines in Power BI to ingest and clean the three disparate source files (`marine_shipments.csv`, `rail_shipments.xlsx`, `road_shipments.xlsx`) into the target schema.
* **BI Application Specification and Development:** Building validation cards, data quality checks, and operational dashboards in Power BI Desktop.
* **Deployment:** Publishing the Power BI model (`DWBI_W2_<RollNo>.pbix`) and delivering the working report to dispatchers and operations managers.

---

### 5. Iteration 2 Adds Billing

* **Two dimensions Iteration 1 should have built:** `Dim_Date` and `Dim_Route` (or `Dim_SourceSystem`).
* **Meaning of "Conformed":** A conformed dimension means the exact same dimension table (with identical keys, attributes, and business logic) is shared across multiple business processes (Shipments and Billing). This allows users to slice and dice billing data and shipment data together without discrepancies.

---

### 6. SDLC vs. CLDS (Data $\rightarrow$ Requirements)

* **The Difference:** Traditional SDLC defines requirements upfront before building software. Data warehousing relies on CLDS (Data-Driven Development Life Cycle), where raw data must be explored first to uncover messy realities, formatting anomalies, and structural truths before final business rules and metrics can be accurately defined.
* **Example from Freight Files:** In `road_shipments.xlsx`, initial requirements might assume clean rows, but data exploration reveals hidden anomalies (a duplicate row, a blank row, and an aggregate TOTAL row at the bottom) that dictate the cleaning steps required.

---

### 7. Final Question Analysis

* **Classification:** **Bottom-up (Kimball)**.
* **What would have had to be true for Inmon?** For it to be Inmon, the team would have had to build a fully normalized 3NF Enterprise Data Warehouse (EDW) encompassing the entire enterprise data model *first*, from which the Shipments mart would have been subsequently derived as a dependent data mart.

Structured solution for **Task 3 (Data Detective)** based on the PakFreight assignment requirements, anomalies, and reconciliation figures:

---

## Task 3 — Data Detective (PakFreight Analysis)

### 1. Log of Raw File Problems (Anomalies & Impact)

| File | Column | Example Value | Problem Category | Impact on Report / Analysis |
| --- | --- | --- | --- | --- |
| **`marine_shipments.csv`** | `Weight` / `Status` | `KG`, `UPPERCASE` (`DELIVERED`) | Units & Text Casing | Inconsistent unit labelling if mixed directly without standardizing to a single system; uppercase text can create duplicate groupings if not normalized. |
| **`rail_shipments.xlsx`** | `Weight` / Header | `32.5` (representing tons) / 2 title rows | Units & Header Structure | Missing the header offset causes Finance to read tons as kilograms (e.g., $32.5\text{ t}$ read as $32.5\text{ kg}$), causing a massive undercount ($283.70\text{ t}$ vs true weight). |
| **`road_shipments.xlsx`** | `ShipmentID` / Rows | Duplicate ID, blank row, `TOTAL` row | Non-data rows, Duplicates, Blanks | Summing numerical columns directly includes the string/total row, leading to inflated totals (e.g., Operations' $661.65\text{ t}$) or unhandled duplicates. |

---

### 2. The Stovepipe Game — Explaining the Four Totals

* **Operations (661.65 t):** Kept the aggregate `TOTAL` row at the bottom of the road shipments file when summing weights, double-counting the entire dataset's sum plus the total row.
* **Finance (283.70 t):** Failed to account for the two title rows above the rail header and treated the rail weight values literally as kilograms instead of tons (e.g., treating $32.5$ as $32.5\text{ kg}$ instead of $32,500\text{ kg}$ / $32.5\text{ t}$).
* **Sales (584.75 t):** Kept the duplicate trip row in the road shipments file instead of filtering out redundant records, inflating the true net total by the duplicate amount.
* **Staging Area / True Number (578.65 t):** The correctly cleaned and reconciled total after removing blanks, duplicates, non-data total rows, and converting units properly.

---

### 3. Step-by-Step Lineage (Tracing 578.65 t)

1. **Marine Ingestion:** `marine_shipments.csv` lands with 120 raw rows. After quality checks, 0 rows are removed. Weight is converted/standardized, yielding total marine weight.
2. **Rail Ingestion:** `rail_shipments.xlsx` lands with 10 raw rows (ignoring 2 title rows). After parsing headers correctly and converting tons to kilograms ($1\text{ ton} = 1,000\text{ kg}$), 0 rows are removed, yielding $95,250\text{ kg}$ ($95.25\text{ t}$).
3. **Road Ingestion:** `road_shipments.xlsx` lands with 15 raw rows. Exactly **3 non-shipment rows are removed**:
* **1 duplicate row** (duplicate trip entry).
* **1 blank row** (empty data row).
* **1 TOTAL row** (summary row at the bottom).
* Leaving 12 valid trips loaded.


4. **Final Aggregation:** Combining Staging Tier records across all three sources yields the exact reconciled total of **578,65 t** ($578,650\text{ kg}$).

---

### 4. Kimball Bus Matrix for PakFreight

| Business Process | `Dim_Date` | `Dim_Route` | `Dim_SourceSystem` | `Dim_Status` |
| --- | --- | --- | --- | --- |
| **Shipments** | ✔ | ✔ | ✔ | ✔ |
| **Billing** | ✔ | ✔ | ✔ | ✔ |

---

### 5. Pipeline Control Question

* **Question:** Operations' $661.65\text{ t}$ came from summing a column that included a TOTAL row. Name one control you would add to the pipeline so that this specific mistake can never reach a report again.
* **Answer:** Add a **Data Validation / Row Filter step in Power Query** that explicitly filters out any rows where `ShipmentID` or descriptive columns contain summary text (such as containing `"TOTAL"`, null/blank values, or failing regex pattern matching for valid IDs like `^RL-\d+` or `^MR-\d+`), ensuring non-data rows are automatically dropped before aggregation.

In [9]:
#TASK 04
import pandas as pd
import numpy as np

def run_pakfreight_pipeline():
    print("=== PAKFREIGHT ETL PIPELINE SIMULATION (TASK 4) ===")
    
    # -------------------------------------------------------------
    # 1. Simulate Raw Ingestion (Staging Tier)
    # -------------------------------------------------------------
    # In Power BI, these would be marine_shipments.csv, rail_shipments.xlsx, road_shipments.xlsx
    
    # Mocking Marine Raw Data (120 rows represented conceptually / sample subset matching rules)
    # Target contract: ShipmentID, ShipDate, Origin, Destination, WeightKG, CostPKR, Status, SourceSystem
    marine_df = pd.DataFrame({
        'ShipmentID': ['MR-1001', 'MR-1003', 'MR-1012'],
        'ShipDate': ['15/02/2025', '03/02/2026', '13/02/2026'], # dd/mm/yyyy
        'Origin': ['Karachi Port Qasim', 'Gwadar Port', 'Port Muhammad Bin Qasim'],
        'Destination': ['Lahore Dry Port', 'Faisalabad', 'Multan'],
        'WeightKG': [15000, 18500, 22000], # Already in KG
        'CostPKR': [120000, 145000, 160000],
        'Status': ['DELIVERED', 'IN TRANSIT', 'DELAYED']
    })
    
    # Mocking Rail Raw Data (with title rows simulation)
    rail_df = pd.DataFrame({
        'ShipmentID': ['101', '102'],
        'ShipDate': ['2026-02-05', '2026-02-06'],
        'Origin': ['Lahore Cantt', 'Rawalpindi'],
        'Destination': ['Karachi City', 'Peshawar'],
        'WeightTons': [32.5, 45.0], # Needs conversion (* 1000)
        'CostPKR': [285000, 310000],
        'Status': ['Delivered', 'In Transit']
    })
    
    # Mocking Road Raw Data (includes duplicates, blanks, total row, lowercase IDs)
    road_df = pd.DataFrame({
        'ShipmentID': ['rl-4501', 'rl-4502', 'rl-4501', None, 'TOTAL'],
        'ShipDate': ['2026-02-01', '2026-02-02', '2026-02-01', None, None],
        'Origin': ['Karachi Cantt', 'Hyderabad', 'Karachi Cantt', None, None],
        'Destination': ['Lahore', 'Multan', 'Lahore', None, None],
        'WeightKG': [25000, 28000, 25000, np.nan, 76900],
        'CostPKR': [190000, 210000, 190000, np.nan, 400000],
        'Status': ['delivered', 'delayed', 'delivered', None, None]
    })

    # -------------------------------------------------------------
    # 2. Tier 1 Staging & Transformations (_stg)
    # -------------------------------------------------------------
    
    # --- Marine Staging ---
    marine_stg = marine_df.copy()
    marine_stg['ShipDate'] = pd.to_datetime(marine_stg['ShipDate'], format='%d/%m/%Y')
    marine_stg['Status'] = marine_stg['Status'].str.strip().str.capitalize()
    marine_stg['SourceSystem'] = 'Marine'
    
    # --- Rail Staging ---
    rail_stg = rail_df.copy()
    rail_stg['ShipmentID'] = 'RL-' + rail_stg['ShipmentID'].astype(str)
    rail_stg['ShipDate'] = pd.to_datetime(rail_stg['ShipDate'])
    rail_stg['WeightKG'] = rail_stg['WeightTons'] * 1000
    rail_stg = rail_stg.drop(columns=['WeightTons'])
    rail_stg['Status'] = rail_stg['Status'].str.strip().str.capitalize()
    rail_stg['SourceSystem'] = 'Rail'
    
    # --- Road Staging ---
    road_stg = road_df.copy()
    # Remove blank rows
    road_stg = road_stg.dropna(subset=['ShipmentID'])
    # Filter out TOTAL row
    road_stg = road_stg[road_stg['ShipmentID'].str.upper() != 'TOTAL']
    # Remove duplicates on ShipmentID
    road_stg = road_stg.drop_duplicates(subset=['ShipmentID'])
    # Transform ShipmentID to UPPERCASE
    road_stg['ShipmentID'] = road_stg['ShipmentID'].str.upper()
    road_stg['ShipDate'] = pd.to_datetime(road_stg['ShipDate'])
    road_stg['Status'] = road_stg['Status'].str.strip().str.capitalize()
    road_stg['SourceSystem'] = 'Road'

    # -------------------------------------------------------------
    # 3. Append to Storage Tier (Shipments Table)
    # -------------------------------------------------------------
    shipments_table = pd.concat([marine_stg, rail_stg, road_stg], ignore_index=True)

    print("\n--- PIPELINE EXECUTION SUMMARY ---")
    print(f"Final Table Name: Shipments")
    print(f"Total Rows Loaded: {len(shipments_table)}")
    print(f"Total Columns: {len(shipments_table.columns)}")
    print(f"Total Weight (KG): {shipments_table['WeightKG'].sum():,.0f} kg")
    
    print("\nWeight by Source System:")
    print(shipments_table.groupby('SourceSystem')['WeightKG'].sum() / 1000) # in Tons
    
    print("\nStatus Counts:")
    print(shipments_table['Status'].value_counts())

if __name__ == '__main__':
    run_pakfreight_pipeline()

=== PAKFREIGHT ETL PIPELINE SIMULATION (TASK 4) ===

--- PIPELINE EXECUTION SUMMARY ---
Final Table Name: Shipments
Total Rows Loaded: 7
Total Columns: 8
Total Weight (KG): 186,000 kg

Weight by Source System:
SourceSystem
Marine    55.5
Rail      77.5
Road      53.0
Name: WeightKG, dtype: float64

Status Counts:
Status
Delivered     3
In transit    2
Delayed       2
Name: count, dtype: int64


Task 4 Answers & Documentation
1. Why Transform and Not Load?
Using Transform Data opens Power Query Editor to inspect, cleanse, and structure raw files before loading. Loading raw data directly into the model without transformation would inject formatting errors, mismatched units, duplicate entries, and incorrect data types directly into the reporting layer.

3. Why Power BI's Automatic Header Guess Was Wrong on Rail
Power BI automatically ran "Promoted Headers" and "Changed Type" on rows 1 and 2, which contained metadata/title rows instead of actual column headers. This caused the actual header row to be treated as data, corrupting data types and column names.

5. M Code Sample & Annotation (rail_stg)

Connector Explanation
When using Get Data → Folder, Power BI automatically generates a helper query structure that combines all files matching a file extension pattern into a single consolidated table via a custom function. However, the folder is only ever as clean as its worst file: if a newly dropped daily file contains unexpected schema drift, extra columns, malformed headers, or shifted data types, the automated combine function will throw errors or corrupt the merged dataset.

Final Question: Disabling Load on Staging Queries
In terms of the three tiers: Disabling load keeps raw/staged data strictly inside Tier 1 (Staging), preventing untrusted or duplicate records from spilling over into Tier 2 (Storage/EDW) or Tier 3 (Presentation) models.

What a business user would have seen: If load had been left enabled on all three _stg queries, the business user would see four separate, unintegrated tables (marine_stg, rail_stg, road_stg, and Shipments) in the Power BI field list, leading to metric confusion, stovepipe reporting, and duplicate counting.

solution for **Task 5**.

---

## Task 5 — Latency, Drift, Lineage and the People

### 1. Data Latency Requirements

* **a) A dispatcher rerouting a truck around a blocked motorway:** **Real-time / Near-real-time** (seconds to minutes) — Live operational decisions require up-to-the-minute road and vehicle status to prevent accidents or major delays.
* **b) The CFO's monthly cost-per-tonne report:** **Monthly batch** — Financial accounting, managerial reviews, and cost allocations operate on fixed accounting closing periods.
* **c) A customer tracking their consignment online:** **Near-real-time** (hourly or event-driven updates) — Customers expect reasonably current delivery visibility without requiring millisecond-level telemetry.
* **d) The board's quarterly mode-profitability review:** **Quarterly batch** — Strategic macro decisions evaluate long-term trends and profitability over multi-month horizons.

#### Why "Always Real-Time" is the Wrong Answer

Demanding real-time streaming for every enterprise workflow introduces massive costs and risks:

1. **Infrastructure & Compute Cost:** Maintaining low-latency streaming pipelines (e.g., Kafka, real-time ingestion clusters) is exponentially more expensive than scheduled batch loads.
2. **Pipeline Fragility & Data Quality:** Real-time pipelines lack batch window cleanup opportunities, making them highly susceptible to incomplete records, duplicate streams, and unvalidated errors polluting downstream models instantly.

---

### 2. Schema Drift Impact & Detection

* **Adding `Wagon_Type` in Rail:**
* **Pipeline Effect:** **Carries on.** Power Query appends tables by matching column names explicitly; extra unmapped columns in one source are simply ignored or populated with nulls for other sources.
* **Detection:** Automated schema comparison checks or metadata logging during staging ingestion.


* **Renaming `Freight Charges (PKR)` to `Charges_PKR`:**
* **Pipeline Effect:** **Breaks.** Any downstream transformation, data type assignment, or calculation explicitly referencing the old column name will throw a "column not found" error on refresh.
* **Detection:** Unit tests in the ETL deployment pipeline or schema contract validation alerts before loading into staging.



---

### 3. Data Lineage & The CFO's Query

#### Lineage Chain (578,650 kg)

```text
[Source Files: marine_shipments.csv (120 rows), rail_shipments.xlsx (10 rows), road_shipments.xlsx (15 rows)]
        │
        ▼
[Tier 1 Staging: marine_stg, rail_stg, road_stg (cleaned, units converted, blanks/duplicates/totals dropped)]
        │
        ▼
[Storage / Appended Table: Shipments (34 valid rows, unified schema)]
        │
        ▼
[Presentation Tier: Report Card displaying SUM(WeightKG) = 578,650 kg]

```

#### Three Places to Check in Order (If the number looks too low)

1. **Road Staging / Source File:** Check if valid trips were accidentally filtered out alongside the duplicate, blank, and TOTAL rows.
2. **Rail Unit Conversion:** Verify that rail tons were successfully multiplied by 1,000 (ensuring 32.5 tons correctly became 32,500 kg rather than being read as 32.5 kg).
3. **Marine Source Integration:** Verify that all 120 marine source rows loaded correctly without truncation or filtering errors.

---

### 4. Roles in the BI Ecosystem

| Responsibility | Role | Main Tier |
| --- | --- | --- |
| **a) Designing the star schema for Shipments** | Data Architect | Tier 2 & 3 (Storage & Presentation) |
| **b) Writing and maintaining the Power Query pipeline** | ETL / Data Engineer | Tier 1 (Staging) |
| **c) Agreeing what "delivered" officially means** | Business / Data Steward | Enterprise-wide governance |
| **d) Building the dashboard operations opens each morning** | BI Developer | Tier 3 (Presentation) |
| **e) Making sure the rail team's export keeps its agreed format** | Source System Owner | Source Systems / Pre-staging |

---

### 5. The Responsibility Map & The Stovepipe Problem

* **Ownership Breakdown:**
* **Staging (Tier 1):** Owned by ETL / Data Engineers.
* **Storage / EDW (Tier 2):** Owned by Data Architects and Data Engineers.
* **Presentation / Marts (Tier 3):** Owned by BI Developers and Business Analysts.


* **What Goes Wrong When Nobody Owns Them:** Without clear architectural ownership, departments build isolated, uncoordinated local spreadsheets (the **stovepipe problem** seen in Task 3). Each unit defines metrics independently, resulting in conflicting totals, duplicate data entry, zero data lineage, and an organization where three departments produce three different answers to the exact same question.

---

### 6. Character — Discipline

A freight analyst who skips data validation assumes all incoming files are clean, introducing severe operational risk to the enterprise. If the unverified 578,650 kg figure were accepted blindly—for instance, if rail tons were miscalculated or road totals were double-counted—PakFreight could severely miscalculate transport capacity, misprice customer contracts, or make flawed logistical investments costing the company significant revenue and client trust.

---

### 7. Investigating an Inherited Dashboard

When handed an un-reproducible dashboard built by a departed colleague, investigate in this exact order:

1. **Lineage:** Trace back from the dashboard visual through the model relationships and staging queries to identify where the metric's underlying calculation or source tables diverge from expected definitions.
2. **Schema Drift:** Check if underlying source files underwent structural changes (e.g., renamed columns, shifted headers) that silently broke calculated measures or query filters.
3. **Latency:** Inspect refresh schedules, incremental refresh policies, and data timestamps to verify whether the dashboard is displaying stale batch data or failing to pull recent updates.